# Introduction

This Notebook intends to explore Point Cloud Manupulation using Open3D.

1. Segmentation
2. Extraction
3. Translation
4. Scaling
5. Global Registration
6. Local Registration

In [1]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

# Import Pyvista and set backend-trame
import pyvista as pv
pv.set_jupyter_backend('trame')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [23]:
# Load and Visualise the SFM point cloud
sfm_ply_path = "Clouds/StanfordBunny_vp2/quad-sfm.ply"
ref_pc = o3d.io.read_point_cloud(sfm_ply_path)
# o3d.visualization.draw_geometries([ref_pc],
#                                   zoom=1,
#                                   front=[0.4257, -0.2125, -0.8795],
#                                   lookat=[0, 0, 0.0],
#                                   up=[-0.0694, -0.9768, 0.2024],
#                                   point_show_normal=True)
o3d.visualization.draw_plotly([ref_pc],
                                  zoom=0.3,
                                  front=[-0.4999, -0.1659, -0.8499],
                                  lookat=[2.1813, 0.619, 2.0999],
                                  up=[0.1204, -1, 0.215])

# Segmentation

Segmentation is performed with the help of DBSCAN an unsupervised clustering algorithm. The parameters to specify in the DBSCAN algorithm include `eps` and `min_points`.

The number of unique elements the from the result of `cluster_dbscan` besides -1 represent the number of clusters. The array assigns a label to each point in the point cloud and the the label is a number indicating the cluster to which the point belongs. A **-1** label denotes a noisy point.

In [83]:
with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Debug) as cm:
    labels = np.array( ref_pc.cluster_dbscan(
        eps=0.2, 
        min_points=10, print_progress=True)
    )
max_label = labels.max()
print(f"point cloud has {max_label + 1} clusters")
colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
ref_pc.colors = o3d.utility.Vector3dVector(colors[:, :3])
o3d.visualization.draw_plotly([ref_pc],
                                  zoom=0.3,
                                  front=[-0.4999, -0.1659, -0.8499],
                                  lookat=[2.1813, 0.619, 2.0999],
                                  up=[0.1204, -1, 0.215])

# Extract Segment
unique_labels = np.unique(labels[labels != -1])

# Create a list to store individual segment point clouds
segment_point_clouds = []

# Extract segments
for label in unique_labels:
    segment_indices = np.where(labels == label)[0]
    segment_points = np.asarray(ref_pc.points)[segment_indices]
    segment_pcd = o3d.geometry.PointCloud()
    segment_pcd.points = o3d.utility.Vector3dVector(segment_points)
    segment_point_clouds.append(segment_pcd)

# o3d.visualization.draw_plotly([ref_pc],
#                               zoom=0.3,
#                               front=[-0.4999, -0.1659, -0.8499],
#                               lookat=[2.1813, 0.619, 2.0999],
#                               up=[0.1204, -1, 0.215])

[Open3D DEBUG] Precompute neighbors.
[Open3D DEBUG] Done Precompute neighbors.
[Open3D DEBUG] Compute Clusters
[Open3D DEBUG] Done Compute Clusters: 2
point cloud has 2 clusters
Precompute neighbors.[========================================] 100%


In [80]:
segment_indices = np.where(labels == -1)[0]
segment_indices
new_pc = np.asarray(ref_pc.points)[segment_indices]
new_pc.shape
segment_pc = o3d.utility.Vector3dVector(new_pc)

o3d.visualization.draw_plotly([segment_pc],
                              zoom=0.3,
                              front=[-0.4999, -0.1659, -0.8499],
                              lookat=[2.1813, 0.619, 2.0999],
                              up=[0.1204, -1, 0.215])

AttributeError: 'open3d.cuda.pybind.utility.Vector3dVector' object has no attribute 'get_geometry_type'

In [33]:
print("Recompute the normal loaded point cloud")
ref_pc.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
ref_pc.orient_normals_towards_camera_location()

Recompute the normal loaded point cloud


PointCloud with 3784 points.

# Scale the Point Cloud

The sparse point cloud from SfM is consistently larger and far away. However, given the intrinsic parameters as well as the multiple viewpoints, a coarse scale of the depth can easily be established. If we consider the point cloud to have a pivot at the origin, scaling the point cloud about this origin will not only reduce the size but also the depth of the point cloud. 

In [78]:
# Define your arbitrary point
arbitrary_point = np.array([0, 0, 0])  # replace x, y, z with your values

# Translate the point cloud so the arbitrary point is at the origin
ref_pc.translate(-arbitrary_point, relative=False)

# Scale the point cloud
scale_factor = 0.5  # replace with your scale factor
ref_pc.scale(scale_factor, center=(0, 0, 0))

# Translate the point cloud back
ref_pc.translate(arbitrary_point, relative=False)

# Now, pcd is the scaled point cloud about the arbitrary point
o3d.visualization.draw_plotly([ref_pc],
                               zoom=0.3,
                                  front=[-0.4999, -0.1659, -0.8499],
                                  lookat=[2.1813, 0.619, 2.0999],
                                  up=[0.1204, -1, 0.215])